# Compare CellViT Inference Runs (Tables + Disagreement Images)

This notebook:
1. Defines a `MODEL_RUNS` dict mapping model labels → run directories.
2. Runs inference for each run directory (optional / can skip if results already exist).
3. Loads each `inference_results.json` and builds:
   - Overall metrics table
   - Per-class PQ table
   - Per-class F1/Precision/Recall table
4. Computes "most different" images between two models (by `bPQ` or `Dice` deltas).

> Tip: run the **inference** step only when your training jobs are finished (or point to already-finished run dirs).


In [8]:
from __future__ import annotations

import json
import math
import os
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [9]:
# ----------------------------
# 1) Configure paths + runs
# ----------------------------

# Set this to the AI-GUIDED-CLEAN root on SCC
ROOT_PATH = Path("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN")

INFER_SCRIPT = ROOT_PATH / "CellViT-plus-plus/cellvit/training/evaluate/inference_cellvit_experiment_pannuke.py"

# Edit this dict for your current experiments
MODEL_RUNS: Dict[str, Path] = {
    # "SAM-H Baseline (Prev)": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local/2025-12-05T174106_tcga_finetune_256",
    "FiLM Rosie Weights SamH Baseline": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-11T233928_FilmRosieWeights-samhbaseline",
    "FiLM Rosie Weights z1z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-11T233928_FilmRosieWeights-z1z4",
    "FiLM Rosie Weights z3z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-11T234426_FilmRosieWeights-z3z4",
    "FiLM Rosie Weights z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-11T233928_FilmRosieWeights-z4",
    "Virchow FiLM Rosie Weights z1z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_virchow_film/2026-02-12_virchow_printdims_seed19/2026-02-12T104940_Virchow-Rosie-FiLM_z1z4",
    "Virchow FiLM Rosie Weights z3z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_virchow_film/2026-02-12_virchow_printdims_seed19/2026-02-12T104940_Virchow-Rosie-FiLM_z3z4",
    "Virchow FiLM Rosie Weights z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_virchow_film/2026-02-12_virchow_printdims_seed19/2026-02-12T104940_Virchow-Rosie-FiLM_z4",
    "Virchow FiLM Rosie Weights baseline": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_virchow/2025-12-06T223702_tcga_finetune_256_virchow",

    
    ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_virchow_film/2026-02-12_virchow_printdims_seed19/2026-02-12T104940_Virchow-Rosie-FiLM_z1z4
}

GPU_ID = 0

In [10]:
# ----------------------------
# 2) Helpers: run inference
# ----------------------------

def run_inference_for_run(run_dir: Path, gpu: int = 0, force: bool = False) -> Path:
    """Run inference script for a single run directory.
    Returns path to inference_results.json.
    """
    run_dir = Path(run_dir)
    out_json = run_dir / "inference_results.json"

    if out_json.exists() and not force:
        print(f"✅ exists, skipping: {out_json}")
        return out_json

    cmd = [
        "python",
        str(INFER_SCRIPT),
        "--run_dir", str(run_dir),
        "--gpu", str(gpu),
    ]

    print("▶️", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not out_json.exists():
        raise FileNotFoundError(f"Expected {out_json} but not found.")
    print(f"✅ wrote: {out_json}")
    return out_json


def run_all_inference(model_runs: Dict[str, Path], gpu: int = 0, force: bool = False) -> Dict[str, Path]:
    results = {}
    for name, rd in model_runs.items():
        print(f"\n=== {name} ===")
        results[name] = run_inference_for_run(rd, gpu=gpu, force=force)
    return results

In [11]:
# ----------------------------
# 3) Helpers: parse json → tables
# ----------------------------

def _safe_float(x):
    try:
        if x is None:
            return float("nan")
        if isinstance(x, str):
            # handle "nan"
            if x.lower() == "nan":
                return float("nan")
        return float(x)
    except Exception:
        return float("nan")


def load_inference_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def build_overall_table(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for model_name, d in results.items():
        ds = d.get("dataset", {})
        row = {"model": model_name}
        for k, v in ds.items():
            row[k] = _safe_float(v)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model")
    # nicer ordering (keep common metrics first if present)
    preferred = ["mPQ", "mDQ", "mSQ", "bPQ", "bDQ", "bSQ", "Binary-Cell-Dice-Mean", "Binary-Cell-Jacard-Mean",
                 "f1_detection", "precision_detection", "recall_detection", "Tissue-Multiclass-Accuracy"]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    return df[cols].sort_index()


def build_perclass_pq_table(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for model_name, d in results.items():
        pq = d.get("nuclei_metrics_pq", {})
        row = {"model": model_name}
        for cls, val in pq.items():
            row[cls] = _safe_float(val)
        rows.append(row)
    return pd.DataFrame(rows).set_index("model").sort_index()


def build_perclass_detection_table(results: Dict[str, dict]) -> pd.DataFrame:
    # nuclei_metrics_d: {class: {f1_cell, prec_cell, rec_cell}}
    rows = []
    for model_name, d in results.items():
        det = d.get("nuclei_metrics_d", {})
        row = {"model": model_name}
        for cls, stats in det.items():
            if not isinstance(stats, dict):
                continue
            row[f"{cls}__f1"] = _safe_float(stats.get("f1_cell"))
            row[f"{cls}__prec"] = _safe_float(stats.get("prec_cell"))
            row[f"{cls}__rec"] = _safe_float(stats.get("rec_cell"))
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model").sort_index()
    return df

In [12]:
# ----------------------------
# 4) Run inference (optional)
# ----------------------------
# If your inference_results.json already exist, keep force=False.
# If you want to rerun (overwrite), set force=True.

FORCE_RERUN = False

# Uncomment to run:
# json_paths = run_all_inference(MODEL_RUNS, gpu=GPU_ID, force=FORCE_RERUN)

In [13]:
# ----------------------------
# 5) Load jsons + build tables
# ----------------------------

def load_all_results(model_runs: Dict[str, Path]) -> Dict[str, dict]:
    out = {}
    for name, rd in model_runs.items():
        p = rd / "inference_results.json"
        if not p.exists():
            raise FileNotFoundError(f"Missing {p}. Run inference first.")
        out[name] = load_inference_json(p)
    return out

results = load_all_results(MODEL_RUNS)

overall_df = build_overall_table(results)
perclass_pq_df = build_perclass_pq_table(results)
perclass_det_df = build_perclass_detection_table(results)

display(overall_df)

,mPQ,mDQ,mSQ,bPQ,bDQ,bSQ,Binary-Cell-Dice-Mean,Binary-Cell-Jacard-Mean,f1_detection,precision_detection,recall_detection,Tissue-Multiclass-Accuracy
model,,,,,,,,,,,,
FiLM Rosie Weights SamH Baseline,0.616701,0.745302,0.744660,0.682231,0.823989,0.820302,0.821330,0.709284,0.853121,0.835359,0.871656,1.0
FiLM Rosie Weights z1z4,0.620897,0.746770,0.765594,0.685578,0.824768,0.830438,0.826611,0.715107,0.843419,0.825559,0.862069,1.0
FiLM Rosie Weights z3z4,0.620193,0.744208,0.751112,0.689638,0.829463,0.820046,0.829954,0.720324,0.848642,0.835481,0.862224,1.0
FiLM Rosie Weights z4,0.612543,0.738344,0.748274,0.679674,0.820086,0.820146,0.819663,0.708115,0.850091,0.837712,0.862842,1.0


## 5b) Styled tables (highlight winners)

These displays highlight:
- **Max value per column** (best metric / best class)
- **Best model row** by a chosen primary metric (e.g., `bPQ` or `mPQ`)


In [14]:
import numpy as np

# Choose a primary metric to define the "best overall" model row highlight.
# Common choices: "bPQ" (binary PQ) or "mPQ" (multi-class PQ)
PRIMARY_METRIC = "bPQ"


def style_highlight_max_per_column(df: pd.DataFrame, primary_metric: str | None = None):
    """Return a pandas Styler that highlights/bolds max per column and (optionally) best row."""
    # base formatting + column-wise max highlight
    sty = (
        df.style
        .format(precision=4)
        .highlight_max(axis=0)
    )

    # bold the max entries per column
    def bold_max(s):
        vals = pd.to_numeric(s, errors="coerce")
        m = np.nanmax(vals.values)
        return ["font-weight: bold;" if (pd.notna(v) and float(v) == m) else "" for v in vals]

    sty = sty.apply(bold_max, axis=0)

    # highlight best row by primary metric
    if primary_metric is not None and primary_metric in df.columns:
        vals = pd.to_numeric(df[primary_metric], errors="coerce")
        if vals.notna().any():
            best_model = vals.idxmax()

            def highlight_best_row(row):
                return ["background-color: rgba(0, 200, 0, 0.12);" if row.name == best_model else "" for _ in row]

            sty = sty.apply(highlight_best_row, axis=1)

    return sty


# Styled overall metrics
display(style_highlight_max_per_column(overall_df, primary_metric=PRIMARY_METRIC))


,mPQ,mDQ,mSQ,bPQ,bDQ,bSQ,Binary-Cell-Dice-Mean,Binary-Cell-Jacard-Mean,f1_detection,precision_detection,recall_detection,Tissue-Multiclass-Accuracy
model,,,,,,,,,,,,
FiLM Rosie Weights SamH Baseline,0.6167,0.7453,0.7447,0.6822,0.8240,0.8203,0.8213,0.7093,0.8531,0.8354,0.8717,1.0000
FiLM Rosie Weights z1z4,0.6209,0.7468,0.7656,0.6856,0.8248,0.8304,0.8266,0.7151,0.8434,0.8256,0.8621,1.0000
FiLM Rosie Weights z3z4,0.6202,0.7442,0.7511,0.6896,0.8295,0.8200,0.8300,0.7203,0.8486,0.8355,0.8622,1.0000
FiLM Rosie Weights z4,0.6125,0.7383,0.7483,0.6797,0.8201,0.8201,0.8197,0.7081,0.8501,0.8377,0.8628,1.0000


In [15]:
# Styled per-class PQ
display(
    perclass_pq_df.style
    .format(precision=4)
    .highlight_max(axis=0)
    .apply(lambda s: ["font-weight: bold;" if pd.notna(v) and float(v) == float(pd.to_numeric(s, errors='coerce').max()) else "" for v in s], axis=0)
)

# Styled per-class detection metrics (F1 / Precision / Recall)
display(
    perclass_det_df.style
    .format(precision=4)
    .highlight_max(axis=0)
    .apply(lambda s: ["font-weight: bold;" if pd.notna(v) and float(v) == float(pd.to_numeric(s, errors='coerce').max()) else "" for v in s], axis=0)
)


,epithelial,lymphocyte,macrophage,neutrophil,other
model,,,,,
FiLM Rosie Weights SamH Baseline,0.6655,0.5617,0.4492,0.4096,nan
FiLM Rosie Weights z1z4,0.6464,0.5722,0.4463,0.4963,nan
FiLM Rosie Weights z3z4,0.6618,0.5416,0.4920,0.4603,nan
FiLM Rosie Weights z4,0.6495,0.5881,0.4408,0.4262,nan


,epithelial__f1,epithelial__prec,epithelial__rec,lymphocyte__f1,lymphocyte__prec,lymphocyte__rec,macrophage__f1,macrophage__prec,macrophage__rec,neutrophil__f1,neutrophil__prec,neutrophil__rec,other__f1,other__prec,other__rec
model,,,,,,,,,,,,,,,
FiLM Rosie Weights SamH Baseline,0.8271,0.8152,0.8393,0.8350,0.8078,0.8640,0.5817,0.5817,0.5817,0.5926,0.7778,0.4786,nan,nan,nan
FiLM Rosie Weights z1z4,0.8058,0.8152,0.7967,0.8199,0.7794,0.8647,0.5533,0.5887,0.5220,0.6103,0.6250,0.5963,nan,nan,nan
FiLM Rosie Weights z3z4,0.8225,0.8278,0.8172,0.8263,0.8005,0.8538,0.5836,0.6096,0.5597,0.5574,0.5037,0.6239,nan,nan,nan
FiLM Rosie Weights z4,0.8234,0.8320,0.8149,0.8335,0.7999,0.8700,0.5584,0.5658,0.5513,0.5938,0.7703,0.4831,nan,nan,nan


In [16]:
# Optional: export styled overall table to HTML (keeps highlights/bold)
OUT_DIR = Path("./comparison_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

html_path = OUT_DIR / "overall_table_styled.html"
(style_highlight_max_per_column(overall_df, primary_metric=PRIMARY_METRIC)
 .to_html(html_path))

print("Wrote:", html_path)


Wrote: comparison_outputs/overall_table_styled.html


In [17]:
# Per-class PQ
display(perclass_pq_df)

# Per-class detection metrics (F1/Prec/Rec)
display(perclass_det_df)

,epithelial,lymphocyte,macrophage,neutrophil,other
model,,,,,
FiLM Rosie Weights SamH Baseline,0.665477,0.561706,0.449159,0.409588,NaN
FiLM Rosie Weights z1z4,0.646392,0.572218,0.446304,0.496267,NaN
FiLM Rosie Weights z3z4,0.661834,0.541575,0.491973,0.460325,NaN
FiLM Rosie Weights z4,0.649475,0.588063,0.440846,0.426222,NaN


,epithelial__f1,epithelial__prec,epithelial__rec,lymphocyte__f1,lymphocyte__prec,lymphocyte__rec,macrophage__f1,macrophage__prec,macrophage__rec,neutrophil__f1,neutrophil__prec,neutrophil__rec,other__f1,other__prec,other__rec
model,,,,,,,,,,,,,,,
FiLM Rosie Weights SamH Baseline,0.827078,0.815203,0.839304,0.834951,0.807787,0.864007,0.581699,0.581699,0.581699,0.592593,0.777778,0.478632,NaN,NaN,NaN
FiLM Rosie Weights z1z4,0.805835,0.815179,0.796703,0.819853,0.779443,0.864684,0.553333,0.588652,0.522013,0.610329,0.625000,0.596330,NaN,NaN,NaN
FiLM Rosie Weights z3z4,0.822482,0.827829,0.817204,0.826325,0.800545,0.853822,0.583607,0.609589,0.559748,0.557377,0.503704,0.623853,NaN,NaN,NaN
FiLM Rosie Weights z4,0.823364,0.831974,0.814931,0.833520,0.799946,0.870035,0.558442,0.565789,0.551282,0.593750,0.770270,0.483051,NaN,NaN,NaN


In [18]:
# Save tables to disk (optional)
OUT_DIR = Path("./comparison_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

overall_df.to_csv(OUT_DIR / "model_overall_metrics.csv")
perclass_pq_df.to_csv(OUT_DIR / "perclass_pq.csv")
perclass_det_df.to_csv(OUT_DIR / "perclass_detection_metrics.csv")

print("Wrote:", OUT_DIR)

Wrote: comparison_outputs


In [20]:
# ----------------------------
# 6) Image-level disagreement
# ----------------------------
# Each inference json has image_metrics: {image_id: {Dice, Jaccard, bPQ, ...}}
# We'll compute top-K images where modelA and modelB differ most by a chosen metric.

def image_metric_df(d: dict, metric: str = "bPQ") -> pd.DataFrame:
    im = d.get("image_metrics", {})
    rows = []
    for image_id, m in im.items():
        if isinstance(m, dict) and metric in m:
            rows.append({"image_id": image_id, metric: _safe_float(m[metric])})
    return pd.DataFrame(rows).set_index("image_id")

def top_disagreement(modelA: str, modelB: str, metric: str = "bPQ", k: int = 20) -> pd.DataFrame:
    a = image_metric_df(results[modelA], metric=metric)
    b = image_metric_df(results[modelB], metric=metric)
    df = a.join(b, lsuffix=f"__{modelA}", rsuffix=f"__{modelB}", how="inner")
    df["delta"] = (df[f"{metric}__{modelB}"] - df[f"{metric}__{modelA}"]).abs()
    df["signed_delta"] = (df[f"{metric}__{modelB}"] - df[f"{metric}__{modelA}"])
    df = df.sort_values("delta", ascending=False).head(k)
    return df

BASELINE_NAME = "FiLM Rosie Weights SamH Baseline"
COMPARE_NAME = "FiLM Rosie Weights z1z4"  # change to any model label
METRIC = "bPQ"

disagree_df = top_disagreement(BASELINE_NAME, COMPARE_NAME, metric=METRIC, k=30)
display(disagree_df)

,bPQ__FiLM Rosie Weights SamH Baseline,bPQ__FiLM Rosie Weights z1z4,delta,signed_delta
image_id,,,,
TCGA-G9-6499-01Z-00-DX1-1_768_0.png,0.000000,0.835896,0.835896,0.835896
TCGA-G7-A8LD-01Z-00-DX1_5_512_256.png,0.000000,0.371344,0.371344,0.371344
TCGA-UZ-A9PO-01Z-00-DX1_4_0_0.png,0.838866,0.484412,0.354453,-0.354453
TCGA-UZ-A9PO-01Z-00-DX1_4_0_512.png,0.599933,0.280395,0.319538,-0.319538
TCGA-G7-A8LD-01Z-00-DX1_8_0_0.png,0.570240,0.880197,0.309958,0.309958
TCGA-KK-A59X-01Z-00-DX1-2_256_0.png,0.769258,0.472856,0.296402,-0.296402
TCGA-SX-A7SR-01Z-00-DX1_2_256_0.png,0.449958,0.743630,0.293672,0.293672
TCGA-G7-A8LD-01Z-00-DX1_8_0_256.png,0.655669,0.885945,0.230276,0.230276
TCGA-G9-6499-01Z-00-DX1-1_256_256.png,0.851782,0.640229,0.211553,-0.211553


## Next step (visualization)
You said you already have a notebook that can visualize a specific image_id across:
- GT
- prediction A
- prediction B

Once you confirm the disagreement table looks good, we can:
1. Take the top-K `image_id`s (best/worst or highest delta),
2. Feed them into your visualization helper to render side-by-side.
